# 💰 Stock Recommender — Step 2: Price Data Fetch & Label Generation

**Goal of this step:**
1. Fetch daily OHLCV prices from `yfinance` for all 49 tickers
2. Implement market-aware date alignment (no look-ahead bias)
3. Compute 1D, 2D, 3D forward returns per article
4. Assign `BUY / HOLD / SELL` labels using region-specific thresholds
5. Analyse label quality and save for Step 3

**Design decisions locked in from Step 1:**

| Decision | Value | Reason |
|---|---|---|
| US threshold | ±1.0% | Best class balance from simulation |
| India threshold | ±0.75% | Lower avg daily volatility on NSE |
| Crypto threshold | ±3.0% | BTC/ETH typical daily swing |
| US T+0 anchor | Same calendar day close | Articles drop 4–9 UTC, NYSE opens 14:30 UTC → pre-market |
| India T+0 anchor | **Next** trading day close | Articles drop 4–9 UTC = 9:30–14:30 IST → during NSE session |
| Crypto T+0 anchor | Same calendar day close | 24/7 trading, no session boundary |

```
Step 1 → EDA & Audit            ✅ DONE
Step 2 → Price Data Fetch       ← YOU ARE HERE
Step 3 → NLP Preprocessing      ← next
Step 4 → FinBERT Features       ← after Step 3
Step 5 → Model Training         ← after Step 4
Step 6 → Evaluation & Tune      ← after Step 5
```


## ⚙️ 0. Install & Imports

In [ ]:
import subprocess, sys
for pkg in ["yfinance", "pandas", "numpy", "matplotlib", "seaborn"]:
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], check=False)

import warnings, time
from pathlib import Path
from datetime import timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import yfinance as yf
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 100)
plt.rcParams["figure.dpi"] = 120
sns.set_style("whitegrid")

print("✅ Imports ready.")
print(f"   yfinance version: {yf.__version__}")


## 📂 1. Load Step 1 Output

In [ ]:
# Load the cleaned, deduplicated dataset produced by Step 1
# If you don't have step1_clean_preview.csv, re-run Step 1 first.

df = pd.read_csv("step1_clean_preview.csv")
df['pub_dt']   = pd.to_datetime(df['published_at'], utc=True, errors='coerce')
df['pub_date'] = pd.to_datetime(df['pub_dt'].dt.date)
df['pub_hour'] = df['pub_dt'].dt.hour
df['pub_dow']  = df['pub_dt'].dt.dayofweek   # 0=Mon, 6=Sun

print(f"Loaded   : {len(df):,} articles, {df['ticker'].nunique()} tickers")
print(f"Regions  : {df['region'].value_counts().to_dict()}")
print(f"Date span: {df['pub_date'].min().date()} → {df['pub_date'].max().date()}")
display(df[['ticker','region','title','pub_dt']].head(4))


## ⚙️ 2. Configuration — Thresholds & Market Hours

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# LABEL THRESHOLDS  (decided from Step 1 simulation)
# ─────────────────────────────────────────────────────────────────────────────
THRESHOLDS = {
    'US'    : 1.00,   # ±1.0% 1-day return
    'IN'    : 0.75,   # ±0.75% — NSE lower daily vol
    'CRYPTO': 3.00,   # ±3.0% — BTC/ETH high vol
}

# ─────────────────────────────────────────────────────────────────────────────
# T+0 ANCHOR LOGIC  (decided from Step 1 hour analysis)
#
#   US stocks  : 4–9 UTC → before NYSE (14:30 UTC) → same calendar day close
#   India      : 4–9 UTC → after NSE open (3:45 UTC) → NEXT trading day close
#   Crypto     : 24/7 → same calendar day close
# ─────────────────────────────────────────────────────────────────────────────
T0_OFFSET = {
    'US'    : 0,   # 0 = same trading day
    'IN'    : 1,   # 1 = shift to next trading day
    'CRYPTO': 0,
}

# yfinance fetch buffer — extra days before first article and after last article
FETCH_BUFFER_DAYS = 10   # covers weekends + public holidays

# Forward return horizons
HORIZONS = [1, 2, 3]   # T+1, T+2, T+3 trading days after T+0

# Multi-horizon label voting weights  (1D most important, 3D least)
HORIZON_WEIGHTS = {1: 0.60, 2: 0.25, 3: 0.15}

print("Configuration locked:")
print(f"  Thresholds : {THRESHOLDS}")
print(f"  T+0 offset : {T0_OFFSET}  (0=same day, 1=next trading day)")
print(f"  Horizons   : {HORIZONS}")
print(f"  Weights    : {HORIZON_WEIGHTS}")


## 📡 3. Fetch OHLCV Data from yfinance

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Build per-ticker fetch windows from the news dataset
# ─────────────────────────────────────────────────────────────────────────────
fetch_plan = (df
    .groupby(['ticker', 'region'])['pub_date']
    .agg(fetch_start='min', fetch_end='max')
    .reset_index()
)
fetch_plan['fetch_start'] = (fetch_plan['fetch_start']
    - pd.Timedelta(days=FETCH_BUFFER_DAYS))
fetch_plan['fetch_end'] = (fetch_plan['fetch_end']
    + pd.Timedelta(days=FETCH_BUFFER_DAYS))

print(f"Tickers to fetch: {len(fetch_plan)}")
print(fetch_plan.to_string(index=False))


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Fetch OHLCV  — one ticker at a time with error handling & progress tracking
# ─────────────────────────────────────────────────────────────────────────────
price_data  = {}    # ticker → DataFrame(Date, Open, High, Low, Close, Volume)
fetch_errors = []

for i, row in fetch_plan.iterrows():
    ticker = row['ticker']
    start  = row['fetch_start'].strftime('%Y-%m-%d')
    end    = row['fetch_end'].strftime('%Y-%m-%d')

    try:
        raw = yf.download(
            ticker,
            start=start,
            end=end,
            interval='1d',
            auto_adjust=True,    # adjusts for splits & dividends
            progress=False,
        )

        if raw.empty:
            print(f"  ⚠️  [{ticker}] No data returned")
            fetch_errors.append({'ticker': ticker, 'error': 'empty response'})
            continue

        # Flatten multi-level columns if present
        if isinstance(raw.columns, pd.MultiIndex):
            raw.columns = raw.columns.get_level_values(0)

        raw = raw[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
        raw.index = pd.to_datetime(raw.index, utc=True).normalize()
        raw.index.name = 'Date'
        raw = raw[~raw.index.duplicated(keep='first')]
        raw = raw.sort_index()

        price_data[ticker] = raw
        print(f"  ✅  [{ticker:18s}] {len(raw):4d} trading days "
              f"| {raw.index.min().date()} → {raw.index.max().date()}")

    except Exception as e:
        print(f"  ❌  [{ticker}] ERROR: {e}")
        fetch_errors.append({'ticker': ticker, 'error': str(e)})

    time.sleep(0.3)   # polite rate limiting

print()
print(f"Successfully fetched : {len(price_data)} / {len(fetch_plan)} tickers")
if fetch_errors:
    print(f"Failed              : {len(fetch_errors)} tickers")
    for err in fetch_errors:
        print(f"  • {err['ticker']}: {err['error']}")


## 🔍 4. Price Data Quality Audit

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Inspect each ticker's price data for:
#   - Missing / zero closes (data gaps)
#   - Extreme single-day moves (possible splits missed)
#   - Price level sanity check
# ─────────────────────────────────────────────────────────────────────────────
quality_rows = []

for ticker, px in price_data.items():
    region = df[df['ticker'] == ticker]['region'].iloc[0]

    close     = px['Close'].dropna()
    daily_ret = close.pct_change().dropna() * 100

    zero_closes  = (px['Close'] == 0).sum()
    null_closes  = px['Close'].isna().sum()
    max_1d_up    = daily_ret.max()
    max_1d_down  = daily_ret.min()
    suspicious   = (daily_ret.abs() > 20).sum()   # >20% in one day

    quality_rows.append({
        'ticker'         : ticker,
        'region'         : region,
        'trading_days'   : len(px),
        'null_closes'    : null_closes,
        'zero_closes'    : zero_closes,
        'max_1d_up_%'    : round(max_1d_up, 2),
        'max_1d_down_%'  : round(max_1d_down, 2),
        'suspicious_days': suspicious,
        'price_min'      : round(close.min(), 2),
        'price_max'      : round(close.max(), 2),
    })

quality_df = pd.DataFrame(quality_rows).sort_values('region')

print("PRICE DATA QUALITY REPORT")
print("=" * 90)
display(quality_df.to_string(index=False))

print()
issues = quality_df[(quality_df['null_closes'] > 0) |
                    (quality_df['zero_closes'] > 0) |
                    (quality_df['suspicious_days'] > 0)]
if len(issues):
    print(f"⚠️  {len(issues)} tickers with data quality issues:")
    display(issues[['ticker','null_closes','zero_closes','suspicious_days']].to_string(index=False))
else:
    print("✅ No data quality issues found.")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Visualise price histories for all tickers (normalized to 100 at start)
# ─────────────────────────────────────────────────────────────────────────────
region_groups = {'US': [], 'IN': [], 'CRYPTO': []}
for ticker, px in price_data.items():
    region = df[df['ticker'] == ticker]['region'].iloc[0]
    region_groups[region].append((ticker, px))

fig, axes = plt.subplots(1, 3, figsize=(22, 6))
region_colors = {'US': 'tab10', 'IN': 'tab20', 'CRYPTO': 'Set1'}

for ax, (region, ticker_list) in zip(axes, region_groups.items()):
    cmap = plt.get_cmap(region_colors[region])
    for j, (ticker, px) in enumerate(ticker_list):
        close = px['Close'].dropna()
        if len(close) < 2:
            continue
        normalized = (close / close.iloc[0]) * 100
        ax.plot(normalized.index, normalized.values,
                linewidth=0.9, alpha=0.65,
                color=cmap(j / max(len(ticker_list), 1)),
                label=ticker)

    ax.set_title(f"{region} — Normalised Price (base=100)", fontweight='bold')
    ax.set_xlabel("Date")
    ax.set_ylabel("Price (indexed to 100)")
    ax.tick_params(axis='x', rotation=30)
    if len(ticker_list) <= 12:
        ax.legend(fontsize=6, loc='upper left', ncol=2)

plt.suptitle("Price History — All Tickers (Normalised)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("step2_price_history.png", bbox_inches='tight', dpi=150)
plt.show()


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Daily return distributions — critical for validating label thresholds
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
region_pal = {'US': '#4CAF50', 'IN': '#2196F3', 'CRYPTO': '#FF5722'}

for ax, (region, ticker_list) in zip(axes, region_groups.items()):
    all_rets = []
    for ticker, px in ticker_list:
        rets = px['Close'].pct_change().dropna() * 100
        all_rets.extend(rets.tolist())

    all_rets = np.array(all_rets)
    thresh   = THRESHOLDS[region]

    ax.hist(all_rets, bins=120, color=region_pal[region],
            edgecolor='white', alpha=0.8, density=True)
    ax.axvline( thresh, color='green', linestyle='--', linewidth=2,
                label=f'+{thresh}% BUY threshold')
    ax.axvline(-thresh, color='red',   linestyle='--', linewidth=2,
                label=f'-{thresh}% SELL threshold')

    buy_pct  = (all_rets >  thresh).mean() * 100
    sell_pct = (all_rets < -thresh).mean() * 100
    hold_pct = 100 - buy_pct - sell_pct

    ax.set_title(
        f"{region} Daily Returns
"
        f"BUY {buy_pct:.1f}%  HOLD {hold_pct:.1f}%  SELL {sell_pct:.1f}%",
        fontweight='bold'
    )
    ax.set_xlabel("1-Day Return (%)")
    ax.set_ylabel("Density")
    ax.set_xlim(np.percentile(all_rets, 0.5), np.percentile(all_rets, 99.5))
    ax.legend(fontsize=8)

    # Print stats
    print(f"{region}  |  std={all_rets.std():.2f}%  "
          f"median={np.median(all_rets):.3f}%  "
          f"BUY={buy_pct:.1f}%  HOLD={hold_pct:.1f}%  SELL={sell_pct:.1f}%")

plt.suptitle("Return Distributions vs Label Thresholds", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("step2_return_distributions.png", bbox_inches='tight', dpi=150)
plt.show()

print()
print("💡 Review the BUY/HOLD/SELL percentages above.")
print("   Ideal target: BUY ≈ SELL ≈ 20–30%, HOLD ≈ 40–60%")
print("   If HOLD > 75%, thresholds need lowering for that region.")


## 📅 5. Market-Aware Date Alignment (Core Logic)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Build a sorted trading-day index per ticker from fetched price data.
# This gives us the ground truth of "which days the market actually traded."
# We use this to look up T+0, T+1, T+2, T+3 correctly — no assumptions about
# calendar days being trading days.
# ─────────────────────────────────────────────────────────────────────────────
trading_calendars = {}    # ticker → sorted array of trading day Timestamps

for ticker, px in price_data.items():
    cal = px.index.sort_values()
    trading_calendars[ticker] = cal

print("Trading calendar samples:")
for t in ['AAPL', 'SBIN.NS', 'BTC-USD']:
    if t in trading_calendars:
        cal = trading_calendars[t]
        print(f"  {t:18s}: {len(cal)} days | "
              f"{cal[0].date()} → {cal[-1].date()}")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Core date-alignment function
#
# Given:  article (ticker, region, pub_date, pub_hour)
# Returns: T+0, T+1, T+2, T+3 as actual trading day Timestamps
#          (or NaT if data unavailable)
#
# Rules:
#   US    : article is pre-market → T+0 = same calendar day
#           (if calendar day is not a trading day, advance to next)
#   India : article is during-session → T+0 = next trading day after pub_date
#   Crypto: T+0 = same calendar day (always a trading day)
# ─────────────────────────────────────────────────────────────────────────────

def find_t0_and_horizons(pub_date, region: str, calendar: pd.DatetimeIndex,
                          t0_offset: int) -> dict:
    # pub_date  : pd.Timestamp  region : US/IN/CRYPTO
    # calendar  : sorted DatetimeIndex of actual trading days
    # t0_offset : 0=same day, 1=next trading day
    pub_date = pd.Timestamp(pub_date).normalize()

    # Convert calendar to tz-naive for comparison
    cal_naive = calendar.normalize().tz_localize(None) if calendar.tz else calendar.normalize()
    pub_naive = pub_date.tz_localize(None) if pub_date.tzinfo else pub_date

    # Find the index of the first trading day >= pub_date
    idx_arr = cal_naive.searchsorted(pub_naive)

    # Exact match: pub_date is a trading day
    if idx_arr < len(cal_naive) and cal_naive[idx_arr] == pub_naive:
        base_idx = idx_arr + t0_offset      # apply offset (0=same, 1=next)
    else:
        # pub_date is a non-trading day (weekend/holiday) — advance to next
        base_idx = idx_arr + t0_offset

    result = {}
    for h in [0, 1, 2, 3]:
        target_idx = base_idx + h
        if 0 <= target_idx < len(cal_naive):
            result[f'T{h}'] = calendar[target_idx]   # return tz-aware original
        else:
            result[f'T{h}'] = pd.NaT

    return result


# ── Quick sanity test ──────────────────────────────────────────────────────────
if 'AAPL' in trading_calendars:
    test_date = pd.Timestamp('2025-09-01', tz='UTC')   # Labour Day (US holiday)
    result = find_t0_and_horizons(test_date, 'US', trading_calendars['AAPL'], T0_OFFSET['US'])
    print("AAPL sanity test (2025-09-01 is US Labour Day):")
    for k, v in result.items():
        print(f"  {k}: {v.date() if pd.notna(v) else 'NaT'}")
    print()

if 'SBIN.NS' in trading_calendars:
    test_date2 = pd.Timestamp('2025-08-15', tz='UTC')  # India Independence Day
    result2 = find_t0_and_horizons(test_date2, 'IN', trading_calendars['SBIN.NS'], T0_OFFSET['IN'])
    print("SBIN.NS sanity test (2025-08-15 is India Independence Day):")
    for k, v in result2.items():
        print(f"  {k}: {v.date() if pd.notna(v) else 'NaT'}")


## 📈 6. Compute Forward Returns

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# For every article in the dataset:
#   1. Determine T+0 (market-aware anchor)
#   2. Compute 1D, 2D, 3D returns:
#        ret_Nd = (Close[T+N] - Close[T+0]) / Close[T+0] * 100
#   3. Store alongside the article
# ─────────────────────────────────────────────────────────────────────────────

records = []
skipped_no_price = 0

for idx, row in df.iterrows():
    ticker = row['ticker']
    region = row['region']
    pub_date = row['pub_date']

    if ticker not in trading_calendars or ticker not in price_data:
        skipped_no_price += 1
        records.append({'idx': idx, 'T0': pd.NaT,
                        'ret_1d': np.nan, 'ret_2d': np.nan, 'ret_3d': np.nan})
        continue

    calendar = trading_calendars[ticker]
    offset   = T0_OFFSET[region]
    dates    = find_t0_and_horizons(pub_date, region, calendar, offset)

    T0, T1, T2, T3 = dates['T0'], dates['T1'], dates['T2'], dates['T3']
    px = price_data[ticker]['Close']

    # Align price index to tz-naive for lookups
    px_naive = px.copy()
    px_naive.index = px_naive.index.tz_localize(None) if px_naive.index.tz else px_naive.index

    def get_close(ts):
        if pd.isna(ts):
            return np.nan
        ts_naive = ts.tz_localize(None) if ts.tzinfo else ts
        ts_norm  = ts_naive.normalize()
        if ts_norm in px_naive.index:
            return px_naive[ts_norm]
        return np.nan

    c0 = get_close(T0)
    c1 = get_close(T1)
    c2 = get_close(T2)
    c3 = get_close(T3)

    def pct_ret(c_start, c_end):
        if np.isnan(c_start) or np.isnan(c_end) or c_start == 0:
            return np.nan
        return (c_end - c_start) / c_start * 100

    records.append({
        'idx'   : idx,
        'T0'    : T0,
        'c0'    : c0,
        'ret_1d': pct_ret(c0, c1),
        'ret_2d': pct_ret(c0, c2),
        'ret_3d': pct_ret(c0, c3),
    })

returns_df = pd.DataFrame(records).set_index('idx')
df = df.join(returns_df[['T0', 'c0', 'ret_1d', 'ret_2d', 'ret_3d']])

total         = len(df)
with_1d       = df['ret_1d'].notna().sum()
with_all3     = df[['ret_1d','ret_2d','ret_3d']].notna().all(axis=1).sum()

print(f"Articles with T+0 close        : {df['c0'].notna().sum():,} / {total:,}")
print(f"Articles with 1D return        : {with_1d:,} / {total:,}  ({with_1d/total*100:.1f}%)")
print(f"Articles with all 3 returns    : {with_all3:,} / {total:,}  ({with_all3/total*100:.1f}%)")
print(f"Skipped (no price data)        : {skipped_no_price:,}")

# Show sample
display(df[['ticker','region','pub_date','T0','c0','ret_1d','ret_2d','ret_3d']].head(10))


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Why do some articles have no returns?
# ─────────────────────────────────────────────────────────────────────────────
missing_ret = df[df['ret_1d'].isna()].copy()

print(f"Articles missing 1D return: {len(missing_ret):,}")
print()
print("Breakdown by region:")
print(missing_ret['region'].value_counts().to_string())
print()
print("Breakdown by ticker:")
print(missing_ret['ticker'].value_counts().head(15).to_string())
print()

# Most recent articles will naturally be missing (no future price yet)
print("Date distribution of missing-return articles:")
missing_by_date = missing_ret.groupby(missing_ret['pub_date'].dt.date).size()
print(missing_by_date.tail(15).to_string())
print()
print("💡 Articles missing returns are almost always the most recent few days")
print("   (no T+1/T+2/T+3 price available yet). These will be excluded from")
print("   training but can be used for live inference.")


## 📊 7. Forward Return Distribution Analysis

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Before labelling, deeply inspect the actual return distributions.
# This validates the thresholds chosen in Step 1.
# ─────────────────────────────────────────────────────────────────────────────
df_labeled = df[df['ret_1d'].notna()].copy()

fig = plt.figure(figsize=(22, 14))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

regions = ['US', 'IN', 'CRYPTO']
region_pal = {'US': '#4CAF50', 'IN': '#2196F3', 'CRYPTO': '#FF5722'}
horizon_labels = ['1-Day (primary)', '2-Day', '3-Day']
ret_cols = ['ret_1d', 'ret_2d', 'ret_3d']

for row_i, region in enumerate(regions):
    sub = df_labeled[df_labeled['region'] == region]
    thresh = THRESHOLDS[region]

    for col_i, (ret_col, hlabel) in enumerate(zip(ret_cols, horizon_labels)):
        ax = fig.add_subplot(gs[row_i, col_i])
        rets = sub[ret_col].dropna()

        if len(rets) == 0:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center')
            continue

        ax.hist(rets, bins=80, color=region_pal[region],
                edgecolor='white', alpha=0.8, density=True)
        ax.axvline( thresh, color='green', linestyle='--', linewidth=2)
        ax.axvline(-thresh, color='red',   linestyle='--', linewidth=2)
        ax.axvline(0,        color='gray',  linestyle='-',  linewidth=0.8, alpha=0.5)

        buy  = (rets >  thresh).mean() * 100
        sell = (rets < -thresh).mean() * 100
        hold = 100 - buy - sell

        ax.set_title(
            f"{region} — {hlabel}
"
            f"B:{buy:.0f}%  H:{hold:.0f}%  S:{sell:.0f}%  "
            f"(n={len(rets):,}  σ={rets.std():.2f}%)",
            fontsize=9, fontweight='bold'
        )
        ax.set_xlabel("Return (%)", fontsize=8)
        ax.set_xlim(rets.quantile(0.005), rets.quantile(0.995))

        # Shade BUY/SELL regions
        xlim = ax.get_xlim()
        ax.axvspan( thresh, max(xlim[1],  thresh+0.1), alpha=0.08, color='green')
        ax.axvspan(min(xlim[0], -thresh-0.1), -thresh,  alpha=0.08, color='red')

plt.suptitle("Forward Return Distributions by Region & Horizon
"
             "(green/red dashed = BUY/SELL thresholds)",
             fontsize=14, fontweight='bold')
plt.savefig("step2_return_eda.png", bbox_inches='tight', dpi=150)
plt.show()


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Correlation between 1D, 2D, 3D returns — how consistent is direction?
# High correlation → single horizon may suffice
# Low correlation  → multi-horizon voting adds real value
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, region in zip(axes, regions):
    sub = df_labeled[df_labeled['region'] == region][['ret_1d','ret_2d','ret_3d']].dropna()
    if len(sub) < 10:
        continue
    corr = sub.corr()
    sns.heatmap(corr, annot=True, fmt='.3f', cmap='RdYlGn',
                center=0, vmin=-1, vmax=1,
                linewidths=0.5, ax=ax,
                xticklabels=['1D','2D','3D'],
                yticklabels=['1D','2D','3D'])
    ax.set_title(f"{region} — Return Horizon Correlations
(n={len(sub):,})",
                 fontweight='bold')

plt.suptitle("Inter-Horizon Return Correlation", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("step2_horizon_correlation.png", bbox_inches='tight', dpi=150)
plt.show()

for region in regions:
    sub = df_labeled[df_labeled['region'] == region][['ret_1d','ret_2d','ret_3d']].dropna()
    if len(sub) > 10:
        r12 = sub['ret_1d'].corr(sub['ret_2d'])
        r13 = sub['ret_1d'].corr(sub['ret_3d'])
        print(f"  {region:6s}  r(1D,2D)={r12:.3f}   r(1D,3D)={r13:.3f}")
print()
print("💡 If correlations are moderate (0.4–0.7), multi-horizon voting adds")
print("   robustness. If near 1.0, using 1D alone is sufficient.")


## 🏷️ 8. Label Generation — Multi-Horizon Weighted Voting

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Strategy: weighted majority vote across 1D, 2D, 3D horizons
#
# For each horizon h:
#   score_h =  +weight_h  if ret_h > +threshold  (BUY vote)
#           =  -weight_h  if ret_h < -threshold  (SELL vote)
#           =   0         otherwise              (HOLD vote)
#
# Final composite score = sum of horizon scores
#   score >  0.25 → BUY
#   score < -0.25 → SELL
#   else          → HOLD
#
# The 0.25 composite threshold means at least the 1D return alone
# must clear the threshold (weight_1d = 0.60 > 0.25) OR two smaller
# horizons must agree.
# ─────────────────────────────────────────────────────────────────────────────

COMPOSITE_BUY_THRESH  =  0.25
COMPOSITE_SELL_THRESH = -0.25

LABEL_MAP = {0: 'SELL', 1: 'HOLD', 2: 'BUY'}

def assign_label(row: pd.Series) -> int:
    region = row['region']
    thresh = THRESHOLDS[region]
    score  = 0.0

    for h, w in HORIZON_WEIGHTS.items():
        ret = row.get(f'ret_{h}d', np.nan)
        if pd.isna(ret):
            continue
        if ret >  thresh:
            score += w      # BUY vote
        elif ret < -thresh:
            score -= w      # SELL vote
        # else HOLD vote (score unchanged)

    if score >  COMPOSITE_BUY_THRESH:
        return 2   # BUY
    elif score < COMPOSITE_SELL_THRESH:
        return 0   # SELL
    else:
        return 1   # HOLD


# Also store individual horizon labels for analysis
def single_horizon_label(ret, threshold) -> int:
    if pd.isna(ret):
        return -1   # unknown
    if ret >  threshold:
        return 2
    if ret < -threshold:
        return 0
    return 1


df_train = df_labeled.copy()
df_train['label']      = df_train.apply(assign_label, axis=1)
df_train['label_1d']   = df_train.apply(
    lambda r: single_horizon_label(r['ret_1d'], THRESHOLDS[r['region']]), axis=1)
df_train['label_2d']   = df_train.apply(
    lambda r: single_horizon_label(r['ret_2d'], THRESHOLDS[r['region']]), axis=1)
df_train['label_3d']   = df_train.apply(
    lambda r: single_horizon_label(r['ret_3d'], THRESHOLDS[r['region']]), axis=1)

# Agreement rate — how often do all three horizons agree?
df_train['horizons_agree'] = (
    (df_train['label_1d'] == df_train['label_2d']) &
    (df_train['label_2d'] == df_train['label_3d']) &
    (df_train['label_1d'] != -1)
)

print("LABEL DISTRIBUTION (composite multi-horizon label)")
print("=" * 55)
total_train = len(df_train)
for lv in [0, 1, 2]:
    cnt = (df_train['label'] == lv).sum()
    bar = '█' * (cnt // 100)
    print(f"  {LABEL_MAP[lv]:4s} ({lv}): {cnt:5,}  ({cnt/total_train*100:5.1f}%)  {bar}")

print()
agree_rate = df_train['horizons_agree'].mean() * 100
print(f"All-horizon agreement rate : {agree_rate:.1f}%")
print(f"(When all 3 horizons give the same signal, label confidence is highest)")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Label distribution deep dive — by region and ticker
# ─────────────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(22, 14))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

label_colors = {0: '#F44336', 1: '#9E9E9E', 2: '#4CAF50'}
label_names  = {0: 'SELL', 1: 'HOLD', 2: 'BUY'}

# ── 1. Overall pie ─────────────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
counts = df_train['label'].value_counts().sort_index()
ax1.pie(counts.values,
        labels=[label_names[k] for k in counts.index],
        autopct='%1.1f%%',
        colors=[label_colors[k] for k in counts.index],
        startangle=90,
        wedgeprops=dict(edgecolor='white', linewidth=2))
ax1.set_title("Overall Label Distribution
(composite multi-horizon)", fontweight='bold')

# ── 2. By region stacked bars ──────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
region_label_df = (df_train.groupby(['region', 'label'])
                   .size().unstack(fill_value=0))
region_label_pct = region_label_df.div(region_label_df.sum(axis=1), axis=0) * 100
region_label_pct.columns = [label_names[c] for c in region_label_pct.columns]
region_label_pct[['SELL','HOLD','BUY']].plot(
    kind='bar', ax=ax2, stacked=True,
    color=['#F44336','#9E9E9E','#4CAF50'], edgecolor='white'
)
ax2.set_title("Label Distribution by Region (%)", fontweight='bold')
ax2.set_xlabel("Region"); ax2.set_ylabel("% of articles")
ax2.tick_params(axis='x', rotation=0)
ax2.legend(title='Signal')

# ── 3. 1D vs composite label agreement heatmap ────────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
agree_matrix = pd.crosstab(
    df_train['label_1d'].map(label_names),
    df_train['label'].map(label_names),
    normalize='index'
) * 100
sns.heatmap(agree_matrix, annot=True, fmt='.1f', cmap='Blues',
            linewidths=0.5, ax=ax3)
ax3.set_title("1D Label vs Composite Label
(row-normalised %)", fontweight='bold')
ax3.set_xlabel("Composite Label"); ax3.set_ylabel("1D Label")

# ── 4. Per-ticker label distribution (top 20) ─────────────────────────────────
ax4 = fig.add_subplot(gs[1, :2])
ticker_label = (df_train.groupby(['ticker','label'])
                .size().unstack(fill_value=0))
ticker_label.columns = [label_names[c] for c in ticker_label.columns]
ticker_label_pct = ticker_label.div(ticker_label.sum(axis=1), axis=0) * 100
ticker_label_pct = ticker_label_pct.sort_values('BUY', ascending=True).tail(25)
ticker_label_pct[['SELL','HOLD','BUY']].plot(
    kind='barh', ax=ax4, stacked=True,
    color=['#F44336','#9E9E9E','#4CAF50'], edgecolor='white', linewidth=0.5
)
ax4.set_title("Label Distribution by Ticker (top 25 by BUY %)", fontweight='bold')
ax4.set_xlabel("% of articles")
ax4.legend(title='Signal', loc='lower right')

# ── 5. High-confidence labels (all horizons agree) ────────────────────────────
ax5 = fig.add_subplot(gs[1, 2])
high_conf = df_train[df_train['horizons_agree']]
hc_counts = high_conf['label'].value_counts().sort_index()
ax5.bar([label_names[k] for k in hc_counts.index],
        hc_counts.values,
        color=[label_colors[k] for k in hc_counts.index],
        edgecolor='white', linewidth=1.5)
ax5.set_title(f"High-Confidence Labels
(all 3 horizons agree, n={len(high_conf):,})",
              fontweight='bold')
ax5.set_ylabel("Count")
for i, (k, v) in enumerate(hc_counts.items()):
    ax5.text(i, v + 20, f'{v:,}', ha='center', fontsize=10)

plt.suptitle("Label Quality Analysis", fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig("step2_label_distribution.png", bbox_inches='tight', dpi=150)
plt.show()


## ✅ 9. Label Quality Checks

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 1. Class imbalance ratio
# ─────────────────────────────────────────────────────────────────────────────
counts = df_train['label'].value_counts().sort_index()
imbalance_ratio = counts.max() / counts.min()

print(f"Class imbalance ratio: {imbalance_ratio:.2f}x  "
      f"(< 3x good, 3–5x manageable, > 5x needs SMOTE)")

# ─────────────────────────────────────────────────────────────────────────────
# 2. Temporal stability — are labels consistent across time?
# ─────────────────────────────────────────────────────────────────────────────
df_train['pub_month'] = df_train['pub_dt'].dt.to_period('M')
monthly_labels = (df_train.groupby(['pub_month','label'])
                  .size().unstack(fill_value=0))
monthly_labels.columns = [label_names[c] for c in monthly_labels.columns]

fig, axes = plt.subplots(2, 1, figsize=(18, 10))

# Stacked area
monthly_pct = monthly_labels.div(monthly_labels.sum(axis=1), axis=0) * 100
monthly_pct = monthly_pct.reindex(columns=['SELL','HOLD','BUY'])
monthly_pct.index = monthly_pct.index.astype(str)
ax = axes[0]
bottom = np.zeros(len(monthly_pct))
for col, color in zip(['SELL','HOLD','BUY'], ['#F44336','#9E9E9E','#4CAF50']):
    if col in monthly_pct.columns:
        ax.bar(monthly_pct.index, monthly_pct[col].values,
               bottom=bottom, color=color, label=col, alpha=0.85, edgecolor='white')
        bottom += monthly_pct[col].fillna(0).values
ax.set_title("Label Distribution Over Time (monthly %)", fontweight='bold')
ax.set_xlabel("Month"); ax.set_ylabel("% of articles")
ax.tick_params(axis='x', rotation=45)
ax.legend(title='Signal')

# BUY% line — key signal for regime detection
buy_pct_monthly = monthly_pct.get('BUY', pd.Series(dtype=float))
sell_pct_monthly = monthly_pct.get('SELL', pd.Series(dtype=float))
axes[1].plot(monthly_pct.index, buy_pct_monthly.values,
             'o-', color='#4CAF50', label='BUY%', linewidth=2, markersize=4)
axes[1].plot(monthly_pct.index, sell_pct_monthly.values,
             's-', color='#F44336', label='SELL%', linewidth=2, markersize=4)
axes[1].axhline(30, color='gray', linestyle='--', linewidth=1, alpha=0.5,
                label='30% reference')
axes[1].set_title("BUY% vs SELL% Over Time
(ideally stable — spikes = market events)",
                  fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)
axes[1].set_ylabel("%"); axes[1].legend()

plt.tight_layout()
plt.savefig("step2_label_temporal.png", bbox_inches='tight', dpi=150)
plt.show()


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3. Per-region breakdown of final label stats
# ─────────────────────────────────────────────────────────────────────────────
print("PER-REGION LABEL SUMMARY")
print("=" * 65)
for region in ['US', 'IN', 'CRYPTO']:
    sub = df_train[df_train['region'] == region]
    c   = sub['label'].value_counts().sort_index()
    n   = len(sub)
    thresh = THRESHOLDS[region]
    imb    = c.max() / c.min() if c.min() > 0 else float('inf')

    print(f"  {region}  (threshold=±{thresh}%)   n={n:,}   imbalance={imb:.1f}x")
    for lv in [0, 1, 2]:
        cnt = c.get(lv, 0)
        print(f"    {label_names[lv]:4s}: {cnt:4,}  ({cnt/n*100:5.1f}%)")
    hc = sub[sub['horizons_agree']]['label'].value_counts().sort_index()
    print(f"    High-confidence count: {hc.sum()} / {n}")
    print()

# ─────────────────────────────────────────────────────────────────────────────
# 4. Return statistics by label — validates label quality
#    BUY articles SHOULD have higher average returns than SELL articles
# ─────────────────────────────────────────────────────────────────────────────
print("RETURN STATISTICS BY LABEL (should increase from SELL → HOLD → BUY)")
print("=" * 65)
ret_by_label = df_train.groupby('label')['ret_1d'].agg(['mean','median','std','count'])
ret_by_label.index = [label_names[i] for i in ret_by_label.index]
print(ret_by_label.round(3).to_string())


## 💾 10. Save Labeled Dataset for Step 3

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Save two files:
#   step2_labeled.csv     → full labeled dataset for NLP pipeline (Step 3)
#   step2_price_cache.pkl → raw price DataFrames (for any Step 3+ feature
#                           that needs lagged prices, volatility etc.)
# ─────────────────────────────────────────────────────────────────────────────
import pickle

# Core output: labeled news articles
save_cols = [
    'id', 'ticker', 'region', 'title', 'summary',
    'source_name', 'source_credibility',
    'published_at', 'pub_date', 'pub_hour', 'pub_dow',
    'T0', 'c0',
    'ret_1d', 'ret_2d', 'ret_3d',
    'label', 'label_1d', 'label_2d', 'label_3d',
    'horizons_agree',
]
df_out = df_train[[c for c in save_cols if c in df_train.columns]].copy()
df_out.to_csv("step2_labeled.csv", index=False)

# Price cache for downstream feature engineering (lagged vol, momentum, etc.)
with open("step2_price_cache.pkl", "wb") as f:
    pickle.dump(price_data, f)

print("Files saved:")
print(f"  step2_labeled.csv      : {len(df_out):,} rows, {df_out.shape[1]} columns")
print(f"  step2_price_cache.pkl  : {len(price_data)} tickers")
print()

# ── Final Step 2 Summary ──────────────────────────────────────────────────────
final_counts = df_out['label'].value_counts().sort_index()
n = len(df_out)
imb = final_counts.max() / final_counts.min()

print("╔══════════════════════════════════════════════════════════════════════╗")
print("║              STEP 2 SUMMARY — PRICE LABELS GENERATED               ║")
print("╠══════════════════════════════════════════════════════════════════════╣")
print(f"║  Tickers fetched          : {len(price_data):>5}                               ║")
print(f"║  Labeled articles         : {n:>6,}                              ║")
print(f"║  Coverage rate            : {n/len(df)*100:>5.1f}% of Step 1 dataset           ║")
print(f"║  Label method             : Multi-horizon weighted vote (1D+2D+3D)║")
print(f"║  Thresholds               : US±1%  IN±0.75%  CRYPTO±3%           ║")
print("╠══════════════════════════════════════════════════════════════════════╣")
for lv in [0, 1, 2]:
    cnt = final_counts.get(lv, 0)
    print(f"║  {label_names[lv]:4s}               : {cnt:>5,}  ({cnt/n*100:5.1f}%)                      ║")
print(f"║  Class imbalance ratio    : {imb:>5.2f}x                             ║")
print("╠══════════════════════════════════════════════════════════════════════╣")
print("║  PLAN FOR STEP 3                                                    ║")
print("║  • NLP preprocessing on title (97% summaries are redundant)        ║")
print("║  • regex cleaning, tokenization, lemmatization                     ║")
print("║  • FinBERT inference → sentiment scores as features                ║")
print("║  • Keyword lexicon signals (bullish/bearish/event)                 ║")
print("║  • Merge all features with price labels from this step             ║")
print("╠══════════════════════════════════════════════════════════════════════╣")
print("║  ✅  Review the label distribution plots above before Step 3       ║")
print("╚══════════════════════════════════════════════════════════════════════╝")
